# Colab 02 — ¿Cuánto tardo en reaccionar, y qué número reporto como incerteza?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 2 — 19/08

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/02_Estadistica_de_una_variable.ipynb)

Mediste tu tiempo de reacción unas cien veces. Tenés cien números distintos y tenés que entregar uno. Este cuaderno es sobre cuál, y sobre todo sobre qué escribir después del ±.

**Al terminar vas a poder:** leer un archivo de datos, construir un histograma con un criterio objetivo para el ancho de bin, distinguir la desviación estándar del error de la media, y escribir un resultado con la cantidad correcta de cifras.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. Leer los datos

Los datos viven en un archivo, no en el cuaderno. Eso no es un detalle
administrativo: es lo que hace que el análisis sea **reproducible** y que
cambiar los datos no implique tocar el código.

In [ ]:
# Esta celda fabrica un archivo de ejemplo para que puedas correr el cuaderno
# hoy mismo, aunque todavía no hayas cargado tus datos.
# Cuando tengas los tuyos, borrala y usá tu archivo.

generador = np.random.default_rng(2026)
ejemplo = generador.normal(loc=0.245, scale=0.032, size=120)
np.savetxt("tiempos_de_reaccion.txt", ejemplo, fmt="%.4f",
           header="tiempo de reaccion (s)")

print("archivo tiempos_de_reaccion.txt creado con 120 mediciones")

In [ ]:
t = np.loadtxt("tiempos_de_reaccion.txt")

print("cantidad de mediciones:", len(t))
print("primeras cinco:", t[:5])
print("mínimo:", t.min(), "  máximo:", t.max())

### 2. El histograma, y por qué el ancho de bin no es una cuestión estética

Un histograma cuenta cuántas mediciones caen en cada intervalo. El problema
es que **la forma que ves depende del ancho del intervalo**, y con datos
reales es perfectamente posible fabricar dos bimodalidades y una gaussiana
con el mismo conjunto, moviendo un solo número.

Por eso el ancho se elige con un criterio, no a ojo. La **regla de Scott**
(Biometrika 66(3), 605, 1979) propone un ancho óptimo bajo el supuesto de
que los datos son aproximadamente normales.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

for ax, nbins in zip(axes, [4, lab.bins_scott(t), 60]):
    ax.hist(t, bins=nbins, edgecolor="black")
    ax.set_xlabel("Tiempo de reacción (s)")
    ax.set_title(f"{nbins} bins")

axes[0].set_ylabel("Cuentas")
plt.show()

print("Scott sugiere", lab.bins_scott(t), "bins")
print("Freedman-Diaconis sugiere", lab.bins_freedman_diaconis(t), "bins")

El de la izquierda oculta la forma; el de la derecha muestra ruido de
conteo y lo hace pasar por estructura. Freedman-Diaconis usa el rango
intercuartil en lugar de la desviación estándar, así que es más robusta si
hay datos anómalos. Cuando difieren mucho, es señal de que hay outliers.

**Regla de conducta:** el criterio de binning se elige y se declara. "Probé
hasta que se vio lindo" no es un criterio.

### 3. El promedio no es una cuenta, es un estimador

Esto parece una sutileza de lenguaje y no lo es.

Cuando calculás $\bar{x} = \frac{1}{N}\sum x_i$ no estás resumiendo tus
datos: estás **estimando** una cantidad que no conocés, el valor verdadero
del que tus mediciones son realizaciones ruidosas. Y como toda estimación,
la tuya tiene una incerteza: si repitieras las cien mediciones mañana, te
daría otro promedio.

El promedio es el mejor estimador posible bajo tres hipótesis: que los
errores son **aleatorios** (no sistemáticos), que las mediciones son
**independientes**, y que la varianza es finita. Ninguna de las tres es
automática. Si tu cronómetro atrasa siempre, la primera falla, y ningún
promedio te salva.

In [ ]:
promedio = np.mean(t)
mediana = np.median(t)

print(f"promedio = {promedio:.5f} s")
print(f"mediana  = {mediana:.5f} s")

Si el promedio y la mediana difieren bastante, la distribución es asimétrica
y el promedio deja de ser el mejor estimador de posición. En tiempos de
reacción es habitual: hay una cola larga hacia los tiempos largos (te
distrajiste) y prácticamente ninguna hacia los cortos (hay un límite
fisiológico). Eso es física del problema, no ruido.

### 4. El error más caro del curso: `np.std` sin `ddof=1`

`np.std` calcula por defecto la desviación estándar **poblacional**,
dividiendo por $N$:

$$\sigma_{\text{pobl}} = \sqrt{\frac{1}{N}\sum (x_i - \bar{x})^2}$$

Pero vos no tenés la población: tenés una muestra, y estimaste el promedio
**con esos mismos datos**. Eso consume un grado de libertad y hace que
dividir por $N$ subestime sistemáticamente la dispersión. Lo correcto es la
desviación estándar **muestral**, dividiendo por $N-1$:

$$s = \sqrt{\frac{1}{N-1}\sum (x_i - \bar{x})^2}$$

En numpy eso se pide con `ddof=1`. No es opcional, y el default está mal
para nuestro uso. Peor todavía: **no da error**, da un número plausible y
chico.

In [ ]:
print(f"np.std(t)           = {np.std(t):.6f} s     <- POBLACIONAL, mal")
print(f"np.std(t, ddof=1)   = {np.std(t, ddof=1):.6f} s     <- muestral, bien")
print(f"diferencia relativa: {100*(1 - np.std(t)/np.std(t, ddof=1)):.2f} %")

Con 120 datos la diferencia es de menos de medio por ciento y podrías pensar
que da igual. Mirá qué pasa cuando medís pocas veces, que es lo habitual.

In [ ]:
print(" N    ddof=0      ddof=1     error relativo")
for n in [3, 5, 10, 30, 120]:
    muestra = t[:n]
    mal = np.std(muestra)
    bien = np.std(muestra, ddof=1)
    print(f"{n:3d}   {mal:.5f}    {bien:.5f}    {100*(1 - mal/bien):5.1f} %")

Con tres mediciones subestimás la dispersión un 18 %. Y tres mediciones es
exactamente lo que se hace en cualquier práctica cuando el tiempo apremia.

### 5. Los dos números que no son el mismo número

Acá está el contenido central de la clase, y probablemente del cuatrimestre.

- $s$, la **desviación estándar muestral**, describe cuánto se dispersa *una
  medición individual*. Es una propiedad del proceso de medición. **No baja
  al medir más veces.**
- $\mathrm{SEM} = s/\sqrt{N}$, el **error estándar de la media**, describe
  cuánto se dispersaría *el promedio* si repitieras el experimento completo.
  Es la incerteza de tu resultado. **Baja como $1/\sqrt{N}$.**

Si tu pregunta es "¿cuánto tarda una persona en reaccionar?", tu resultado es
el promedio y su incerteza es el SEM. Si tu pregunta es "¿cuánto puede llegar
a tardar la próxima vez?", la respuesta la da $s$.

In [ ]:
promedio, s, sem = lab.estadisticos(t)

Verifiquémoslo con tus propios datos: tomamos los primeros $n$ y vemos cómo
evolucionan los dos números.

In [ ]:
enes = np.arange(5, len(t) + 1)
s_vs_n = np.array([np.std(t[:n], ddof=1) for n in enes])
sem_vs_n = s_vs_n / np.sqrt(enes)

fig, ax = plt.subplots()
ax.plot(enes, s_vs_n, "o", ms=3, label="s (dispersión de una medición)")
ax.plot(enes, sem_vs_n, "s", ms=3, label="SEM (incerteza del promedio)")
ax.plot(enes, s / np.sqrt(enes), "crimson", lw=1.5, label=r"$s/\sqrt{N}$")
ax.set_xlabel("Cantidad de mediciones usadas, N")
ax.set_ylabel("Tiempo (s)")
ax.legend()
plt.show()

La serie de arriba fluctúa alrededor de un valor y se queda ahí: medir más
no te hace más consistente. La de abajo baja, y baja despacio. Para dividir
tu incerteza por dos, tenés que **cuadruplicar** el número de mediciones. Esa
es la respuesta cuantitativa a "¿cuántas veces mido?" y es el tema de la
próxima clase.

### 6. Escribir el resultado

Ahora sí, la regla completa de redondeo:

- La **incerteza** se redondea a **una cifra significativa**, salvo que la
  primera sea 1 o 2, en cuyo caso se conservan dos.
- El **valor** se redondea a la **misma posición decimal** que la incerteza.

Referencias: Taylor secc. 2.5; GUM (JCGM 100:2008) secc. 7.2.6. La función
`lab.reportar` lo hace por vos.

In [ ]:
lab.reportar(promedio, sem, "s", nombre="tiempo de reacción")

print()
print("Ejemplos de la regla:")
for valor, error in [(24.783941, 1.2837), (9.8123, 0.052), (0.24537, 0.0029)]:
    print(f"  {valor} ± {error}   ->   {lab.formatear(valor, error)}")

### 7. ¿Y si hay un dato claramente raro?

Aparece uno de 0,9 s porque te distrajiste. La tentación de borrarlo es
enorme y la respuesta honesta es incómoda: **un dato anómalo es primero una
hipótesis sobre un error identificable, y solo después un problema
estadístico**. Si podés señalar qué pasó, se descarta y se informa. Si no,
el criterio de Chauvenet (Taylor cap. 6) da una regla objetiva: se descarta
un dato si, en una muestra de $N$, se esperaba **menos de medio dato** tan
alejado como él.

Las tres reglas de conducta que acompañan al criterio:

1. Se fija **antes** de mirar los datos.
2. Se aplica **una sola vez** (no se itera hasta que quede lindo).
3. **Todo descarte se informa** en el informe, con su justificación.

La versión más honesta de "descartar" es volver a medir ese punto.

In [ ]:
t_con_intruso = np.append(t, 0.91)

print("Chauvenet sobre el conjunto con un dato agregado a mano:")
conservar = lab.chauvenet(t_con_intruso)

print()
print(f"promedio con el intruso : {t_con_intruso.mean():.5f} s")
print(f"promedio sin el intruso : {t_con_intruso[conservar].mean():.5f} s")

### 8. Ejercicios

1. Rehacé todo el cuaderno con **tus** datos. El único cambio necesario es
   borrar la celda que fabrica el archivo de ejemplo y subir el tuyo con el
   ícono de carpeta del panel izquierdo.
2. Superponé sobre tu histograma una línea vertical en el promedio y dos
   bandas en $\bar{x} \pm s$. ¿Qué fracción de tus datos cae adentro?
   (Deberían ser aproximadamente dos tercios; en la Clase 3 vemos por qué.)
3. Dividí tus mediciones en las primeras 50 y las últimas 50. ¿Los dos
   promedios son parecidos? Si el segundo es sistemáticamente menor,
   aprendiste durante el experimento, y eso viola la hipótesis de
   independencia.
4. Contestá por escrito, en dos renglones: **¿reportás $s$ o SEM, y por qué?**

In [ ]:
# Espacio de trabajo para los ejercicios.

### Entrega corta 1 (una página)

Histograma con criterio de bin declarado, $\bar{x}$, $s$, SEM, el resultado
escrito con la regla de redondeo, y la respuesta al ejercicio 4.